# SVR in-depth analysis

Converted from [t_half/9_svr.py](t_half/9_svr.py)

This notebook keeps the original training, evaluation, diagnostics, model export, and applicability-domain workflow.

Although, this notebook has in final section structural visualization. 


So far beyond just basic model train and eval, but still lots more to do to be credible.

 - freeze this version as 9a, then continued in 9b_svr_level_up.py

In [ ]:
import sys
import json
from pathlib import Path

notebookdir = Path.cwd().parents[2]
sys.path.append(str(notebookdir))  # this way we can import src modules even in different subfolders
import joblib
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sqlalchemy as sa
from scipy import stats
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import cross_validate, learning_curve
from sqlalchemy.orm import sessionmaker
from src.db_schema import *
from src.db_utils import get_selected_data
from src.log_utils import log_section, log_to_file
from src.ml_tools import (
    PreprocessedData,
    Preprocessor,
    applicability_domain_leverage,
    chemical_space_morgan_pca,
    chemical_space_pca,
    output_metrics,
    output_metrics_w_return,
    svr_grid_search,
    t_t_split,
)

matplotlib.use("Agg")
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 250)

In [ ]:
# =============================================================================
# USER CONFIGURATION - Change these values as needed
# =============================================================================
compartment = "air"  # Options: air, water, sediment, soil
DATA_SOURCE = "hsbd"  # Options: hsbd, vega (vega has no air data)

target_column = "T_half_days"
use_outlier_removal = True
n_folds = 5
test_size = 0.25
random_state = 42

# =============================================================================
# INTERNAL SETUP - Do not change unless you know what you're doing
# =============================================================================
# Validate configuration
if DATA_SOURCE == "vega" and compartment == "air":
    raise ValueError("VEGA dataset does not contain air compartment data.")

# Paths and DB setup
WORK_DIR = Path.cwd().parent.parent
DATA_DIR = WORK_DIR / "processed_data"
DATABASE_FILE = DATA_DIR / "hsbd_t_half_all.db"
DATABASE_FILE = DATA_DIR / "vega_t_half_soil_water_sediment.db" if DATA_SOURCE == "vega" else DATA_DIR / "hsbd_t_half_all.db"

ENGINE = sa.create_engine(f"sqlite:///{DATABASE_FILE}")
Session = sessionmaker(bind=ENGINE)

# Model output paths
models_dir = WORK_DIR / "models"
models_dir.mkdir(exist_ok=True)
model_file = models_dir / f"SVR_{compartment}.joblib"

# Logging setup
log_time = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
logs_dir = WORK_DIR / "logs" / f"{compartment}_{DATA_SOURCE}" / log_time
logs_dir.mkdir(parents=True, exist_ok=True)
log_file = str(logs_dir / "9_SVR.log")

log_to_file("SVR Model and Analysis", log_file)
log_to_file(f"Notebook run started. Working dir: {WORK_DIR}", log_file)
log_section("Preprocessor class", log_file)

print(f"Log file: {log_file}")

In [ ]:
# Load and preprocess data
data_to_use = get_selected_data(compartment, Session)

log_to_file(
    f"Dataset X and y shapes before preprocessing: X={data_to_use.shape[0]}, y={data_to_use.shape[0]}",
    log_file,
)

to_drop = ["None"]
log_to_file(f"Columns to drop: {to_drop}", log_file)

preprocessor = Preprocessor(
    target_column=target_column,
    to_drop=to_drop,
    remove_outliers=use_outlier_removal,
)

data_clean = preprocessor.preprocess(data_to_use)
preprocessed_data = PreprocessedData(
    name="AirData",
    df=data_clean,
    remove_outliers=use_outlier_removal,
    X=preprocessor.X,
    y_log=preprocessor.y_log,
)

for key, value in preprocessed_data.__dict__.items():
    if key in {"name", "remove_outliers"}:
        log_to_file(f"{key}: {value}", log_file)

log_to_file(
    (f"Dataset X and y shapes after preprocessing: X={preprocessed_data.X.shape[0]}, y_log={preprocessed_data.y_log.shape[0]}"),
    log_file,
)

X = preprocessed_data.X
y = preprocessed_data.y_log

print(f"Preprocessed X shape: {X.shape}")
print(f"Preprocessed y shape: {y.shape}")

In [ ]:
# Train, cross-validate, and evaluate model
log_section("Model section", log_file)

print(f"Performing {n_folds}-fold cross-validation...")
log_to_file(f"Performing {n_folds}-fold cross-validation", log_file, include_timestamp=True)

X_train, X_test, y_train, y_test = t_t_split(X, y, test_size=test_size, random_state=random_state)
smiles_train, smiles_test = preprocessor.get_smiles_for_split(X_train, X_test)

svr, svr_params = svr_grid_search(X_train, y_train)
print(f"Best hyperparameters from grid search: {svr_params}")
log_to_file(f"Best hyperparameters from grid search: {svr_params}", log_file, include_timestamp=True)

cv_scores = cross_validate(
    svr,
    X_train,
    y_train,
    cv=n_folds,
    scoring=["r2", "neg_mean_squared_error", "neg_mean_absolute_error"],
    return_train_score=True,
)

print(f"R2 (CV): {cv_scores['test_r2'].mean():.4f} (+/- {cv_scores['test_r2'].std():.4f})")
print(
    "MSE (CV): "
    f"{-cv_scores['test_neg_mean_squared_error'].mean():.4f} (+/- {cv_scores['test_neg_mean_squared_error'].std():.4f})"
)
print(
    "MAE (CV): "
    f"{-cv_scores['test_neg_mean_absolute_error'].mean():.4f} (+/- {cv_scores['test_neg_mean_absolute_error'].std():.4f})"
)

log_to_file(f"R2 (CV): {cv_scores['test_r2'].mean():.4f} (+/- {cv_scores['test_r2'].std():.4f})", log_file)
log_to_file(
    "MSE (CV): "
    f"{-cv_scores['test_neg_mean_squared_error'].mean():.4f} (+/- {cv_scores['test_neg_mean_squared_error'].std():.4f})",
    log_file,
)
log_to_file(
    "MAE (CV): "
    f"{-cv_scores['test_neg_mean_absolute_error'].mean():.4f} (+/- {cv_scores['test_neg_mean_absolute_error'].std():.4f})",
    log_file,
)

svr.fit(X_train, y_train)
y_pred_svr = svr.predict(X_test)

y_test_exp_svr = np.power(10, y_test)
y_pred_exp_svr = np.power(10, y_pred_svr)

print("Held-out test set performance:")
output_metrics(y_test_exp_svr, y_pred_exp_svr)

results = {
    test_size: {
        "y_test_exp_svr": y_test_exp_svr,
        "y_pred_exp_svr": y_pred_exp_svr,
        "params": svr_params,
        "cv_scores": cv_scores,
    }
}

best_test_size = test_size
params = results[best_test_size]["params"]
cv_scores = results[best_test_size]["cv_scores"]

y_test_exp_svr = results[best_test_size]["y_test_exp_svr"]
y_pred_exp_svr = results[best_test_size]["y_pred_exp_svr"]

print("=" * 60)
print("FINAL RESULTS SUMMARY")
print("=" * 60)
print(f"Test size: {best_test_size}")
print(f"SVR hyperparameters: {params}")
print(f"R2 (CV): {cv_scores['test_r2'].mean():.4f} (+/- {cv_scores['test_r2'].std():.4f})")
output_metrics(y_test_exp_svr, y_pred_exp_svr)
print("=" * 60)

log_to_file("=" * 60, log_file)
log_to_file("FINAL RESULTS SUMMARY", log_file)
log_to_file(f"Test size: {best_test_size}", log_file)
log_to_file(f"SVR hyperparameters: {params}", log_file)
log_to_file(f"R2 (CV): {cv_scores['test_r2'].mean():.4f} (+/- {cv_scores['test_r2'].std():.4f})", log_file)
log_to_file(f"{output_metrics_w_return(y_test_exp_svr, y_pred_exp_svr)}", log_file)
log_to_file("=" * 60, log_file)

In [ ]:
# Learning curve analysis
print("Performing learning curve analysis...")
log_to_file("Learning Curve Analysis", log_file, include_timestamp=True)

train_sizes = np.linspace(0.1, 1.0, 10)
train_sizes_abs, train_scores, val_scores = learning_curve(
    svr,
    X,
    y,
    train_sizes=train_sizes,
    cv=n_folds,
    scoring="r2",
    random_state=random_state,
    n_jobs=-1,
)

train_scores_mean = train_scores.mean(axis=1)
train_scores_std = train_scores.std(axis=1)
val_scores_mean = val_scores.mean(axis=1)
val_scores_std = val_scores.std(axis=1)

print(f"Training sizes: {train_sizes_abs}")
print(f"Validation R2 scores: {val_scores_mean}")
print(f"Training R2 (final): {train_scores_mean[-1]:.4f} (+/- {train_scores_std[-1]:.4f})")
print(f"Validation R2 (final): {val_scores_mean[-1]:.4f} (+/- {val_scores_std[-1]:.4f})")

if len(val_scores_mean) >= 3:
    recent_improvement = val_scores_mean[-1] - val_scores_mean[-3]
    if recent_improvement > 0.01:
        interpretation = "Curves are still improving - more data may help"
    elif abs(train_scores_mean[-1] - val_scores_mean[-1]) > 0.1:
        interpretation = "High variance - model may be overfitting"
    else:
        interpretation = "Curves have converged - more data unlikely to help significantly"
    print(f"Interpretation: {interpretation}")
    log_to_file(f"Interpretation: {interpretation}", log_file)

plt.figure(figsize=(10, 6))
plt.plot(train_sizes_abs, train_scores_mean, "o-", color="royalblue", label="Training score", linewidth=2, markersize=6)
plt.fill_between(
    train_sizes_abs,
    train_scores_mean - train_scores_std,
    train_scores_mean + train_scores_std,
    alpha=0.2,
    color="royalblue",
)
plt.plot(
    train_sizes_abs,
    val_scores_mean,
    "o-",
    color="darkorange",
    label="Cross-validation score",
    linewidth=2,
    markersize=6,
)
plt.fill_between(
    train_sizes_abs,
    val_scores_mean - val_scores_std,
    val_scores_mean + val_scores_std,
    alpha=0.2,
    color="darkorange",
)
plt.xlabel("Training Set Size", fontsize=12)
plt.ylabel("R2 Score", fontsize=12)
plt.title("Learning Curve: SVR Model Performance vs Training Data Size", fontsize=14)
plt.legend(loc="lower right", fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
learning_curve_path = logs_dir / f"learning_curve_{compartment}.png"
plt.savefig(learning_curve_path)
print(f"Learning curve plot saved to {learning_curve_path}")
log_to_file(f"Learning curve plot saved to {learning_curve_path}", log_file)
plt.close()

plt.figure(figsize=(10, 6))
gap = train_scores_mean - val_scores_mean
plt.plot(train_sizes_abs, gap, "o-", color="crimson", linewidth=2, markersize=6)
plt.fill_between(train_sizes_abs, 0, gap, alpha=0.3, color="crimson")
plt.xlabel("Training Set Size", fontsize=12)
plt.ylabel("Training-Validation Gap (R2)", fontsize=12)
plt.title("Overfitting Analysis: Gap Between Training and Validation Scores", fontsize=14)
plt.grid(True, alpha=0.3)
plt.axhline(y=0.05, color="gray", linestyle="--", label="5% gap threshold")
plt.legend(loc="upper right", fontsize=10)
plt.tight_layout()
overfit_path = logs_dir / f"overfitting_gap_{compartment}.png"
plt.savefig(overfit_path)
print(f"Overfitting gap plot saved to {overfit_path}")
log_to_file(f"Overfitting gap plot saved to {overfit_path}", log_file)
plt.close()

In [ ]:
# Save model artifacts and parameter JSON
joblib.dump(svr, model_file)
print(f"Model saved to {model_file}")
log_to_file(f"Model saved to {model_file}", log_file, include_timestamp=True)

cv_rmse_log10 = np.sqrt(-cv_scores["test_neg_mean_squared_error"].mean())
cv_rmse_log10_std = np.sqrt(cv_scores["test_neg_mean_squared_error"].std())

test_mse_log10 = mean_squared_error(y_test, y_pred_svr)
test_rmse_log10 = np.sqrt(test_mse_log10)
test_mae_log10 = mean_absolute_error(y_test, y_pred_svr)

test_rmse_days = np.sqrt(mean_squared_error(y_test_exp_svr, y_pred_exp_svr))
test_mae_days = mean_absolute_error(y_test_exp_svr, y_pred_exp_svr)

print("Additional metrics:")
print(f"CV RMSE (log10): {cv_rmse_log10:.4f} (+/- {cv_rmse_log10_std:.4f})")
print(f"Test RMSE (log10): {test_rmse_log10:.4f}")
print(f"Test MAE (log10): {test_mae_log10:.4f}")
print(f"Test RMSE (days): {test_rmse_days:.2f}")
print(f"Test MAE (days): {test_mae_days:.2f}")

print("Feature information:")
print(f"Original features (X): {X.shape[1]}")
print(f"Training features (X_train): {X_train.shape[1]}")
print(f"Features removed as constants: {X.shape[1] - X_train.shape[1]}")

model_parameters = {
    "model_name": "SVR",
    "compartment": compartment,
    "target_column": target_column,
    "model_params": params,
    "feature_columns": list(X_train.columns),
    "y_value_transformation": "log10",
    "test_size": best_test_size,
    "cross-validation_folds": n_folds,
    "model_performance_scores": {
        "cv_r2_mean": cv_scores["test_r2"].mean(),
        "cv_r2_std": cv_scores["test_r2"].std(),
        "cv_mse_mean": -cv_scores["test_neg_mean_squared_error"].mean(),
        "cv_mse_std": cv_scores["test_neg_mean_squared_error"].std(),
        "cv_rmse_log10_mean": cv_rmse_log10,
        "cv_rmse_log10_std": cv_rmse_log10_std,
        "cv_mae_mean": -cv_scores["test_neg_mean_absolute_error"].mean(),
        "cv_mae_std": cv_scores["test_neg_mean_absolute_error"].std(),
        "test_rmse_log10": test_rmse_log10,
        "test_mae_log10": test_mae_log10,
        "test_rmse_days": test_rmse_days,
        "test_mae_days": test_mae_days,
    },
    "outlier_removal": use_outlier_removal,
    "preprocessing_drops": to_drop,
    "n_samples_train": len(X_train),
    "n_samples_test": len(X_test),
    "n_features": X_train.shape[1],
}

params_file = model_file.with_suffix(".json")
with open(params_file, "w") as f:
    json.dump(model_parameters, f, indent=2)

print(f"Parameters saved to {params_file}")
log_to_file(f"Parameters saved to {params_file}", log_file, include_timestamp=True)

In [ ]:
# Diagnostics, chemical-space analysis, residual analysis

# plot true vs predicted with MAE region
mae = mean_absolute_error(y_test_exp_svr, y_pred_exp_svr)

plt.figure(figsize=(7, 7))
plt.scatter(y_test_exp_svr, y_pred_exp_svr, alpha=0.6)
plt.plot(
    [min(y_test_exp_svr), max(y_test_exp_svr)],
    [min(y_test_exp_svr), max(y_test_exp_svr)],
    "r--",
    label="Ideal",
)
plt.fill_between(
    [min(y_test_exp_svr), max(y_test_exp_svr)],
    [min(y_test_exp_svr) + mae, max(y_test_exp_svr) + mae],
    [min(y_test_exp_svr) - mae, max(y_test_exp_svr) - mae],
    color="gray",
    alpha=0.2,
    label=f"MAE +/- {mae:.2f}",
)
plt.xlabel("True y (days)")
plt.ylabel("Predicted y (days)")
plt.title("SVR: True vs Predicted y with MAE region")
plt.legend()
plt.grid(True)
true_vs_pred_path = logs_dir / f"true_vs_pred_{compartment}.png"
plt.savefig(true_vs_pred_path)
print(f"True vs Predicted plot saved to {true_vs_pred_path}")
log_to_file(f"True vs Predicted plot saved to {true_vs_pred_path}", log_file)
plt.close()

plt.figure(figsize=(7, 7))
mask = (y_test_exp_svr <= 100) & (y_pred_exp_svr <= 100)
plt.scatter(y_test_exp_svr[mask], y_pred_exp_svr[mask], alpha=0.6)
plt.plot([0, 100], [0, 100], "r--", label="Ideal")
plt.fill_between([0, 100], [mae, 100 + mae], [-mae, 100 - mae], color="gray", alpha=0.2, label=f"MAE +/- {mae:.2f}")
plt.xlim(0, 100)
plt.ylim(0, 100)
plt.xlabel("True y (days)")
plt.ylabel("Predicted y (days)")
plt.title("SVR: True vs Predicted y with MAE region (y <= 100)")
plt.legend()
plt.grid(True)
true_vs_pred_100_path = logs_dir / f"true_vs_pred_100_{compartment}.png"
plt.savefig(true_vs_pred_100_path)
print(f"True vs Predicted (<=100) plot saved to {true_vs_pred_100_path}")
log_to_file(f"True vs Predicted (<=100) plot saved to {true_vs_pred_100_path}", log_file)
plt.close()

In [ ]:
# plot feature importance
result = permutation_importance(
    svr,
    X_test,
    y_test,
    n_repeats=10,
    random_state=42,
    scoring="neg_mean_squared_error",
)

importances = result.importances_mean
num_of_feats = 20
indices = np.argsort(importances)[::-1][:num_of_feats]

plt.figure(figsize=(10, 5))
plt.bar(range(num_of_feats), importances[indices])
plt.xticks(range(num_of_feats), X_test.columns[indices], rotation=90)
plt.title(f"Top {num_of_feats} SVR Feature Importances (Permutation)")
plt.ylabel("Mean Importance")
plt.tight_layout()
feat_imp_path = logs_dir / f"feature_importance_{compartment}.png"
plt.savefig(feat_imp_path)
print(f"Feature importance plot saved to {feat_imp_path}")
log_to_file(f"Feature importance plot saved to {feat_imp_path}", log_file)
plt.close()

# chemical space coverage, PCA
pca, scaler, train_pcs, test_pcs, coverage_pct = chemical_space_pca(X_train, X_test, logs_dir, compartment)
log_to_file(f"Test coverage inside training bounding box: {coverage_pct:.1f}%", log_file)
log_to_file(
    f"PCA explained variance: PC1={pca.explained_variance_ratio_[0]:.3f}, PC2={pca.explained_variance_ratio_[1]:.3f}",
    log_file,
)

# applicability domain analysis, Williams leverage method
y_pred_train = svr.predict(X_train)
ad_results = applicability_domain_leverage(
    X_train,
    X_test,
    y_train,
    y_test,
    y_pred_train,
    y_pred_svr,
    logs_dir / "ad_analysis",
    compartment,
)

log_section("Applicability Domain Analysis", log_file)
log_to_file(f"h* threshold : {ad_results['h_star']:.4f}", log_file)
log_to_file(f"Train inside AD : {ad_results['pct_train_inside']:.1f}%", log_file)
log_to_file(f"Test  inside AD : {ad_results['pct_test_inside']:.1f}%", log_file)

cluster_stats = chemical_space_morgan_pca(
    smiles_train,
    smiles_test,
    ad_results,
    logs_dir / "ad_analysis",
    compartment,
)
log_to_file(f"AD-outside test compounds (n={cluster_stats['n_outside_ad']})", log_file)
if "n_clusters" in cluster_stats:
    log_to_file(
        f"Butina clusters: {cluster_stats['n_clusters']}, "
        f"singletons: {cluster_stats['n_singletons']} ({cluster_stats['singleton_pct']:.1f}%)",
        log_file,
    )

# Q-Q plots, and residuals vs predicted and features
residuals = y_test_exp_svr - y_pred_exp_svr

plt.figure(figsize=(8, 5))
sns.scatterplot(x=y_pred_exp_svr, y=residuals)
plt.axhline(0, color="red", linestyle="--")
plt.xlabel("Predicted Values")
plt.ylabel("Residuals")
plt.title("Residuals vs Predicted")
resid_vs_pred_path = logs_dir / f"residuals_vs_pred_{compartment}.png"
plt.savefig(resid_vs_pred_path)
print(f"Residuals vs Predicted plot saved to {resid_vs_pred_path}")
log_to_file(f"Residuals vs Predicted plot saved to {resid_vs_pred_path}", log_file)
plt.close()

plt.figure()
sns.histplot(residuals, kde=True)
plt.title("Residual Distribution")
resid_hist_path = logs_dir / f"residual_hist_{compartment}.png"
plt.savefig(resid_hist_path)
print(f"Residual histogram plot saved to {resid_hist_path}")
log_to_file(f"Residual histogram plot saved to {resid_hist_path}", log_file)
plt.close()

plt.figure()
stats.probplot(residuals, dist="norm", plot=plt)
plt.title("Q-Q Plot of Residuals")
qqplot_path = logs_dir / f"qqplot_residuals_{compartment}.png"
plt.savefig(qqplot_path)
print(f"Q-Q plot of residuals saved to {qqplot_path}")
log_to_file(f"Q-Q plot of residuals saved to {qqplot_path}", log_file)
plt.close()

for col in X_test.columns[indices]:
    plt.figure(figsize=(6, 4))
    sns.scatterplot(x=X_test[col], y=residuals)
    plt.axhline(0, color="red", linestyle="--")
    plt.title(f"Residuals vs {col}")
    resid_vs_feat_path = logs_dir / f"residuals_vs_{col}_{compartment}.png"
    plt.savefig(resid_vs_feat_path)
    print(f"Residuals vs {col} plot saved to {resid_vs_feat_path}")
    log_to_file(f"Residuals vs {col} plot saved to {resid_vs_feat_path}", log_file)
    plt.close()

print("SVR model training and analysis complete. Check log file for details.")

In [ ]:
from rdkit import Chem
from rdkit.Chem import Draw


def draw_compound_structures(df, n=20, random=False):
    """Draws the first n compound structures from the DataFrame. if random bool, then random n compounds are drawn."""

    if random:
        smiles_list = df.sample(n, random_state=42).tolist()
    else:
        # Get the first n SMILES strings
        smiles_list = df.head(n).tolist()

    mols = [Chem.MolFromSmiles(smiles) for smiles in smiles_list]
    img = Draw.MolsToGridImage(mols, molsPerRow=5, subImgSize=(200, 200))
    display(img)

In [ ]:
import ipywidgets as widgets
from IPython.display import display
import random

# smiles_train, smiles_test
draw_compound_structures(smiles_train, n=20, random=True)

In [ ]:
# Interactive viewer for train/test compound structures
N_DISPLAY = 20

# Dispose prior viewer so rerunning this cell keeps a single live panel.
_prev = globals().get("_svr_viewer")
if _prev is not None:
    _prev["dropdown"].unobserve(_prev["on_change"], names="value")
    _prev["refresh_btn"].on_click(_prev["on_refresh"], remove=True)
    _prev["dropdown"].close()
    _prev["refresh_btn"].close()
    _prev["info_label"].close()
    _prev["ui"].close()

smiles_data = {"Train": smiles_train, "Test": smiles_test}

dropdown = widgets.Dropdown(
    options=["Train", "Test"],
    value="Train",
    description="Set:",
    layout=widgets.Layout(width="150px"),
)
refresh_btn = widgets.Button(
    description="Refresh",
    button_style="info",
    layout=widgets.Layout(width="100px"),
)
info_label = widgets.Label()

viewer_state = {"image_handle": None}


def render(_=None):
    dataset = dropdown.value
    smiles_series = smiles_data[dataset]
    n = len(smiles_series)
    smiles_list = smiles_series.tolist()

    if n > N_DISPLAY:
        chosen = random.sample(smiles_list, N_DISPLAY)
        info_label.value = f"{dataset}: {n:,} compounds - showing {N_DISPLAY} random"
    else:
        chosen = smiles_list
        info_label.value = f"{dataset}: {n:,} compounds - showing all"

    mols = [Chem.MolFromSmiles(s) for s in chosen]
    img = Draw.MolsToGridImage(mols, molsPerRow=5, subImgSize=(350, 200))

    if viewer_state["image_handle"] is None:
        viewer_state["image_handle"] = display(img, display_id=True)
    else:
        viewer_state["image_handle"].update(img)


def on_change(change):
    render(change["new"])


def on_refresh(_):
    render()


ui = widgets.HBox([dropdown, refresh_btn, info_label])
display(ui)
render()

dropdown.observe(on_change, names="value")
refresh_btn.on_click(on_refresh)

globals()["_svr_viewer"] = {
    "dropdown": dropdown,
    "refresh_btn": refresh_btn,
    "info_label": info_label,
    "ui": ui,
    "on_change": on_change,
    "on_refresh": on_refresh,
}